# ゼロから作る Deep Learning ❸ 輪読会
## 第1ステージ「微分を自動で求める」 ― ステップ 1 〜 5

**はじめに**：  
* Deep Learning のフレームワークは微分を求めるためのツールとも言える。  
* Pytorch のようなフレームワークを再現するためには、微分を自動的に求める機能が必要。  
* 自動的に微分を求めるため、「変数」と「関数」を表す 2 つのクラスが必要とされる。  

**このステージの目標**：  
* `y.backward()` を実行するだけで、微分 `dy/dx` 自動的に計算される仕組みを作る。  
* PyTorch などでも中心的な役割を果たす自動微分 (automatic differentiation) を実装する。  

```python
x = Variable(np.array(0.5))
y = square(exp(square(x)))  # y = (exp(x^2))^2
y.backward()
print(x.grad)               # dy/dx を自動で求める
```

> **Table of contents**
> 1. ステップ 1：箱としての変数 (`Variable`)
> 2. ステップ 2：変数を生み出す関数 (`Function`)
> 3. ステップ 3：関数の連結 (合成関数)
> 4. ステップ 4：数値微分
> 5. ステップ 5：バックプロパゲーションの理論

## ライブラリの読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print("NumPy version:", np.__version__)


---
# ステップ  1：箱としての変数

## 1.1 「変数」とは

プログラミングの入門書とどうように、DeZero でも変数を「データを入れる箱」だと考える。  
このイメージの重要な点は以下の 3 つである。  

- 箱とデータは別物である (箱そのものは中身ではない)
- 箱にはデータが入る (= 代入)
- 箱の中を覗けばデータが分かる (= 参照)

なぜわざわざ「箱」を用意するか？  
単に数値を扱うだけなら NumPy の配列で十分だが、後のステップで、この箱に 「微分の値 (grad)」 や 「この箱を生み出した関数 (creator)」 といった追加情報を持たせていきたい。従って、そのようないくつもの情報をひとまとめに管理するための「入れ物」が必要となる。今はまだ何も入っていない空の箱を作るところから始める。  


## 1.2 `Variable` クラスの実装

変数は英語で variable なので、DeZero の変数を `Variable` クラスとして実装する。

```python
class Variable:
    def __init__(self, data):
        self.data = data
```

`__init__` は Python でインスタンスが作られるときに自動で呼ばれる初期化メソッドである。そこでは、与えられた引数 `data` をインスタンス変数と呼ばれる `self.data` にそのまま格納しているだけ。これだけで「箱」としての役割を果たす。実際のデータは `self.data` に保持されることになる。  

> **補足**：`self` は「そのインスタンス自身」を指す。`self.data = data` は「この箱の `data` という棚に、受け取ったデータをしまう」という意味


In [ ]:
class Variable:
    def __init__(self, data):
        self.data = data  # 受け取ったデータを箱（インスタンス変数 data）にしまうだけ


# 実行例 ------------------------------------------------------------
data = np.array(1.0)  # NumPy の多次元配列 (ここでは 0 次元 = スカラ)
x = Variable(data)    # data を箱に入れる
print("x.data =", x.data)  # 箱の中を確認 (参照)


上記の `x` は `Variable` インスタンス (実体) であり、実際の数値は `x.data` の中にある。`x` はデータそのものではなく、「データを持つ存在＝箱」だというイメージ。

箱の中身は後から入れ替えることもできます。`x.data = ...` と書けば新しいデータが代入されます。


In [ ]:
# 箱の中身を入れ替える (代入)
x.data = np.array(2.0)
print("x.data =", x.data)


## 1.3 【補足】NumPy の多次元配列

機械学習のシステムは、基礎となるデータ構造に多次元配列を使う。そのため DeZero の `Variable` も、NumPy の多次元配列 (`numpy.ndarray`、以降 `ndarray` と略す) を入れる箱として設計する。  

多次元配列とは、数値などの要素が規則的に並んだデータ構造である。要素の並びには「方向」があり、その方向を「次元」や「軸」と呼ぶ。  

| 呼び名 | 数学の呼び名 | 次元数 (`ndim`) | 例 |
|---|---|---|---|
| 0次元配列 | スカラ | 0 | `1` |
| 1次元配列 | ベクトル | 1 | `[1, 2, 3]` |
| 2次元配列 | 行列 | 2 | `[[1,2,3],[4,5,6]]` |

また、多次元配列はテンソルとも呼ばれる。`ndarray` の `ndim` 属性 (number of dimensions の略) で次元数を確認できる。  


In [ ]:
# NumPy 多次元配列の次元数を確認する
a = np.array(1.0)          # スカラ (0 次元)
b = np.array([1, 2, 3])    # ベクトル (1 次元)
c = np.array([[1, 2, 3],
              [4, 5, 6]])  # 行列 (2 次元)

print("a:", a, "-> ndim =", a.ndim)  # 0
print("b:", b, "-> ndim =", b.ndim)  # 1
print("c:\n", c, "\n-> ndim =", c.ndim, ", shape =", c.shape)  # 2, (2, 3)


> **ステップ 1 のまとめ**
> - `Variable` は「データを入れる箱」であり、実際のデータは `.data` に入る。
> - DeZero では、NumPy の `ndarray` を扱い、スカラ・ベクトル・行列はすべて `ndarray` で表せる (= テンソル)。
> - 現在は 3 行しかない Variable クラスだが、これを出発点に本格的なフレームワークを実装する。


---
# ステップ 2：変数を生み出す関数

## 2.1 「関数」とは

ステップ 1 で「変数（箱）」を実装した。次は、その変数に何らかの処理を施す「関数」を作る。  

ここでいう関数とは、数学の関数 $y = f(x)$ と同じイメージである。ある変数 $x$ を受け取って、別の変数 $y$ を生み出すものである。DeZero では、これを計算グラフの形で理解する。  

$$ \textcircled{x} \;\longrightarrow\; \boxed{f} \;\longrightarrow\; \textcircled{y} $$

- ○（まる）= 変数：`Variable` インスタンス
- □（しかく）= 関数：これから作る `Function` インスタンス

> 計算グラフとは、計算を「ノード (節点)」と「エッジ (矢印)」で表したデータ構造 (およびその図) である。後のステップで、この計算グラフをたどることで自動微分を実現する。  


## 2.2 `Function` クラスの実装

`Function` クラスを設計するときには次の 2 点に注意する。  

1. 入力も出力も `Variable` インスタンスにすること (箱を受け取り、箱を返す)
2. 実際のデータは `Variable` の `.data` に入っていること

この方針で `Function` クラスを書くと、次のようになる。  

```python
class Function:
    def __call__(self, input):
        x = input.data        # 1. 箱からデータを取り出す
        y = self.forward(x)   # 2. 具体的な計算 (forward に委譲)
        output = Variable(y)  # 3. 結果を箱に詰める
        return output

    def forward(self, x):
        raise NotImplementedError()  # 継承先で実装させる
```

ここでのポイントは 2 つの特殊なメソッドにある。  

- `__call__`：Python の特殊メソッドで、これを定義すると `f = Function()` としたとき、`f(x)` と書けば `__call__` メソッドを呼び出せる。つまり、インスタンスを「関数のように」呼べるようになる。役割は「箱からデータを取り出す → 計算する → 箱に詰め直す」という共通の流れ。  
- `forward`：ここが実際の計算の中身である。`Function` 自身では、`NotImplementedError` (=「ここは継承先で書いてね」という意思表示) を出すだけにして、具体的な計算は子クラスに任せている。  

このように「共通の流れは親クラス (`Function`)、個別の計算は子クラス」と役割を分けるのが、フレームワーク設計の定石である。  


In [ ]:
class Function:
    def __call__(self, input):
        x = input.data        # 1. Variable (箱) から実データを取り出す
        y = self.forward(x)   # 2. 具体的な計算は forward メソッドに任せる
        output = Variable(y)  # 3. 計算結果を Variable (箱) に詰め直す
        return output

    def forward(self, x):
        # この基底クラスでは中身を実装せず、例外を出す。
        # 「forward は継承先で実装すべき」というメッセージになる。
        raise NotImplementedError()


次に、この `Function` を継承して、具体的な関数を作る。ここでは「入力を 2 乗する」関数 `Square` ($y = x^2$) を実装する。  

`Square` は `Function` を継承しているので、`__call__` はそのまま受け継がれる。やることは `forward` に「2 乗する」という中身を書くだけ。  


In [ ]:
class Square(Function):
    def forward(self, x):
        return x ** 2  # 受け取った ndarray を要素ごとに 2 乗する


## 2.3 `Function` クラスを使う

実際に `Square` を使ってみる。`Variable` の `x` (中身は 10) を `Square` インスタンスに入力する。  

各変数に何が入るかを意識しながら確認する。  

- `x`：`Variable` インスタンス（`x.data` は `10`）
- `f`：`Square` インスタンス
- `y`：`f(x)` の戻り値。これも `Variable` インスタンスで、`y.data` は `100`


In [ ]:
x = Variable(np.array(10))
f = Square()
y = f(x)  # f.__call__(x) が呼ばれる

print("y の型 :", type(y))  # <class '...Variable'> = 箱が返ってきている
print("y.data :", y.data)   # 100 = 10 の 2 乗


狙いどおり、`Square` を通すと「箱 `x`」から「箱 `y`」が生み出され、その中身が 2 乗されている。入力も出力も `Variable` で統一されていることが、次のステップ (関数の連結) で重要になる。  

> ステップ 2 のまとめ
> - `Function` は「箱を受け取り、計算し、箱を返す」基底クラス。共通処理を `__call__` に、個別計算を `forward` に分離している。  
> - `__call__` を定義すると、インスタンスを `f(x)` のように関数として呼べる。  
> - `Square(Function)` のように継承して、具体的な関数を定義できる。  


---
# ステップ 3：関数の連結

## 3.1 Exp 関数の実装

関数を 1 種類だけ作っても面白くありません。もう 1 つ、指数関数 $y = e^x$ を実装する。ここで $e$ はネイピア数 ($e = 2.718\ldots$) である。  

$$ y = e^{x} $$

やることは `Square` のときと同じで、`Function` を継承して `forward` の中身を `np.exp(x)` に変えるだけ。  


In [ ]:
class Exp(Function):
    def forward(self, x):
        return np.exp(x)  # e^x を計算 (np.exp は NumPy の指数関数)


## 3.2 関数を連結する

`Function` の `__call__` は入力も出力も `Variable` である。ということは、ある関数の出力 (箱) を、そのまま次の関数の入力 (箱) にできる。これを使えば複雑な関数を表現できる。  

例として、次の合成関数を計算してみる。  

$$ y = \left( e^{\,x^{2}} \right)^{2} $$

これを「2乗 → 指数 → 2乗」の 3 段階に分解する。途中の値も含めて、各変数に何が入るかを整理すると次のようになる。  

| 変数 | 中身 | 計算 |
|---|---|---|
| `x` | 入力 | $x = 0.5$ |
| `a` | `A(x)` | $a = x^2$ |
| `b` | `B(a)` | $b = e^{a}$ |
| `y` | `C(b)` | $y = b^2$ |

計算グラフで書くと、変数 (○) と関数 (□) が交互に並ぶ一直線の形になる。  

$$ x \to \boxed{A=\text{Square}} \to a \to \boxed{B=\text{Exp}} \to b \to \boxed{C=\text{Square}} \to y $$


In [ ]:
A = Square()  # 1 段目：2 乗
B = Exp()     # 2 段目：指数
C = Square()  # 3 段目：2 乗

x = Variable(np.array(0.5))
a = A(x)  # a = x^2    ：a も Variable (a.data に値)
b = B(a)  # b = exp(a) ：b も Variable
y = C(b)  # y = b^2    ：y も Variable

# 途中経過も確認する
print("x.data =", x.data)
print("a.data =", a.data, "  (= x^2)")
print("b.data =", b.data, "  (= exp(a))")
print("y.data =", y.data, "  (= b^2 = (exp(x^2))^2)")


ここで重要なのは、途中に登場する `x`, `a`, `b`, `y` がすべて `Variable` インスタンスだという点である。`Function` の入出力が `Variable` で統一されているおかげで、こうして自然に関数を連結できる。  

複数の関数を順に適用して作られるこの大きな変換を合成関数と呼ぶ。単純な計算 (2 乗、指数) でも、連結すれば複雑な計算が表現できるという点が重要。  

> ステップ 3 のまとめ
> - 入出力を `Variable` で統一したので、関数を連結できる。  
> - 連結してできる大きな関数が「合成関数」。  
> - 次のステップでは、この合成関数の微分を求める。  


---
# ステップ 4：数値微分

ステップ 3 で合成関数を計算できるようになった。いよいよ本題の微分に入る。まずは一番シンプルな方法である数値微分をもちいて微分を計算する。  

## 4.1 微分とは

微分とは、ざっくり言えば「変化の割合」のこと。  

- ある物体の「位置」を時刻で微分すると → 速度
- 「速度」を時刻で微分すると → 加速度

このように、微分は「ある量が、別の量のごくわずかな変化に対してどれだけ変化するか」を表す。数式では、関数 $f(x)$ の $x$ における微分は次のように定義される。  

$$ f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h} \tag{4.1} $$

**この式の意味：**

- $\dfrac{f(x+h) - f(x)}{h}$ は、$x$ と $x+h$ という近い2点を結ぶ直線の傾きである (分子が「$y$ の変化量」、分母が「$x$ の変化量 $h$」)。  
- $\lim_{h \to 0}$（リミット）は「$h$ を限りなく 0 に近づける」という操作です。  
- 2 点の間隔 $h$ をどんどん狭めていくと、その傾きは「点 $x$ における接線の傾き」に近づく。これが微分 $f'(x)$ である。  

$f'(x)$ もまた $x$ の関数になっており、これを $f(x)$ の導関数と呼ぶ。  


## 4.2 数値微分の実装

式(4.1) をそのままコンピュータで計算したいが、問題が 1 つある。コンピュータは「極限」を扱えず、$h \to 0$ をそのまま実行できない。  

そこで、$h$ を「十分に小さい具体的な値」(たとえば $h = 0.0001 = $ `1e-4`) で近似する。このように微小な差分で変化量を求める手法を数値微分 (numerical differentiation) という。  

さらに精度を上げる工夫として、中心差分近似を使う。式 (4.1) は $x$ と $x+h$ の 2 点を使う「前進差分」だが、代わりに $x-h$ と $x+h$ の 2 点を使う。  

$$ f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \tag{中心差分} $$

なぜ中心差分の方が良いのか？  

直感的には、$x$ を中心に左右対称に点を取ることで、近似の誤差 (傾きのずれ) が打ち消し合うためである。厳密にはテイラー展開で証明できる (前進差分の誤差が $O(h)$ なのに対し、中心差分は $O(h^2)$ と小さい)。分母が $2h$ になっている点 (左右合わせて幅 $2h$ ぶん動いた) に注意。  

これを `numerical_diff(f, x, eps=1e-4)` という関数で実装する。`eps` (イプシロン) は微小な値 $h$ のこと。  


In [ ]:
def numerical_diff(f, x, eps=1e-4):
    # f：微分対象の関数 (Function インスタンス、または関数)
    # x：微分する点 (Variable インスタンス)
    # eps：微小な値 h
    x0 = Variable(x.data - eps)  # x - h
    x1 = Variable(x.data + eps)  # x + h
    y0 = f(x0)                   # f(x - h)
    y1 = f(x1)                   # f(x + h)
    return (y1.data - y0.data) / (2 * eps)  # 中心差分：(f(x+h) - f(x-h)) / 2h


まずは単純な $y = x^2$ の $x = 2.0$ における微分を求める。  
解析的に解くと $\dfrac{dy}{dx} = 2x$ なので、$x=2.0$ では正確な値は 4.0 になる。数値微分でこれにどれだけ近づくか確認する。  


In [ ]:
f = Square()
x = Variable(np.array(2.0))
dy = numerical_diff(f, x)
print("数値微分の結果 :", dy)
print("解析解 (2x)   :", 2 * 2.0)


`4.000000000004...` のように、正確な値 `4.0` にごく近い値が得られた。わずかな誤差は近似によるもの。  

## 4.3 合成関数の微分

数値微分の強みは、どんなに複雑な合成関数でも同じやり方で微分できること。ステップ 3 の合成関数  

$$ y = \left( e^{\,x^{2}} \right)^{2} $$

を、関数 `f` としてまとめて定義し、その微分を $x=0.5$ で求めてみる。  

Python では関数もオブジェクトなので、関数 `f` を `numerical_diff` の引数として渡せる。  


In [ ]:
def f(x):
    A = Square()
    B = Exp()
    C = Square()
    return C(B(A(x)))  # y = (exp(x^2))^2

x = Variable(np.array(0.5))
dy = numerical_diff(f, x)
print("dy/dx =", dy)


結果は約 `3.297...`。これは「$x$ を 0.5 から微小に変化させると、$y$ はその変化量の約 3.297 倍だけ変化する」という意味。  

ついに、目的の計算をコードで書けば微分が自動で求まる状態にたどり着いた。では、なぜわざわざ次のステップでバックプロパゲーションを学ぶのか？

## 4.4 数値微分の問題点

数値微分には問題が 2 つ存在する  

1. 誤差が含まれる (精度の問題)
   近い値どうしの引き算で桁落ちが起きる。たとえば有効桁 4 桁で `1.234 - 1.233 = 0.001` のように、有効桁数が一気に減ってしまう。  

2. 計算コストが高い (速度の問題)
   変数 1 つの微分を求めるのに、関数の計算 (順伝播) を 2 回行う必要がある。ニューラルネットワークではパラメータが数百万〜数十億個にもなる。各パラメータごとに 2 回の計算を繰り返すのは現実的ではない。  

この 2 つを一気に解決するのが、次のステップのバックプロパゲーションである。  

> 数値微分の意義：  
> 実装が簡単で、おおよそ正しい値が求まる。そのため、後でバックプロパゲーションを実装したときに「その結果が正しいか」を確かめる検算 (勾配確認, gradient checking) に使える。  


### 追加検証①：数値微分の `eps` と誤差の関係

「$h$（`eps`）は小さければ小さいほど正確になるのでは？」と思われるかもしれない。しかし、実際には $h$ を小さくしすぎると桁落ち (丸め誤差) のせいで逆に誤差が増える。合成関数 $y=(e^{x^2})^2$ の $x=0.5$ で、`eps` を変えながら誤差を測ってみる (真の微分は $4(e^{x^2})^2 x \approx 3.2974$)。  

縦軸・横軸ともに対数スケールで見ると、ある最適な `eps` (だいたい $10^{-6}$ あたり) で誤差が最小になり、それより小さくすると丸め誤差で逆に誤差が増える様子が見てとれる。これは数値計算における普遍的な現象である。  

> ちなみに、$y=x^2$ のような 2 次関数だと、中心差分の近似誤差が理論上ゼロになってしまい、この「近似誤差と丸め誤差のせめぎ合い」が観察できない。そこであえて非線形性の強い合成関数を例にしている。


In [ ]:
# eps を変えながら、y=(exp(x^2))^2 の x=0.5 における数値微分の誤差を測る
def f_comp(xv):
    A, B, C = Square(), Exp(), Square()
    return C(B(A(xv)))  # y = (exp(x^2))^2

x0 = 0.5
b0 = np.exp(x0 ** 2)
true_grad = 4 * (b0 ** 2) * x0       # 解析解：dy/dx = 4 b^2 x

eps_list = np.logspace(-1, -13, 30)  # 1e-1 〜 1e-13 まで対数的に
errors = []
for eps in eps_list:
    g = numerical_diff(f_comp, Variable(np.array(x0)), eps=eps)
    errors.append(abs(g - true_grad))

plt.figure(figsize=(7, 4.5))
plt.loglog(eps_list, errors, 'o-')
plt.xlabel("eps (h)")
plt.ylabel("absolute error")
plt.title("Numerical differentiation: error vs eps  (y=(exp(x^2))^2)")
plt.grid(True, which="both", alpha=0.3)
plt.gca().invert_xaxis()  # 左から右へ eps が小さくなるように
plt.show()

best_i = int(np.argmin(errors))
print(f"誤差が最小になる eps = {eps_list[best_i]:.1e}, そのときの誤差 = {errors[best_i]:.2e}")


見てのとおり、`eps` を小さくしていくと最初は誤差が減るが、`1e-6` あたりを境に逆に誤差が増えていく。これは桁落ち (丸め誤差) の影響である。「近似誤差 ($h$ が大きいと増える)」と「丸め誤差 ($h$ が小さいと増える)」のせめぎ合いで、ちょうど良い `eps` が存在する。デフォルトの `1e-4` は、このバランスが取れた無難な選択になっている。  


---
# ステップ 5：バックプロパゲーションの理論

数値微分の 2 大問題 (精度・計算コスト) を解決するのがバックプロパゲーション (誤差逆伝播法) である。微分を効率良く、しかもより正確に求められる。  

このステップは理論の説明のみで、実装は次のステップ (次回の輪読会) で行う。

## 5.1 チェインルール (連鎖律)

鍵となるのがチェインルール (連鎖律, chain rule) である。チェイン (chain) は「鎖」の意味で、複数の関数が鎖のように連結している様子を表す。  

チェインルールとは、合成関数の微分はそれを構成する各関数の微分の積に分解できるという法則である。  

具体例で説明する。$y = F(x)$ という関数が、次の 3 つの関数の合成だとする。  

$$ a = A(x), \quad b = B(a), \quad y = C(b) $$

計算グラフは一直線である。  

$$ x \to \boxed{A} \to a \to \boxed{B} \to b \to \boxed{C} \to y $$

このとき、$x$ に関する $y$ の微分はチェインルールにより次のように書ける。  

$$ \frac{dy}{dx} = \frac{dy}{db}\,\frac{db}{da}\,\frac{da}{dx} \tag{5.1} $$

式の意味：  
「全体の変化率」は「各段の変化率の掛け算」で求まる、ということ。$x$ がちょっと動くと $a$ が $\frac{da}{dx}$ 倍動き、それで $b$ が $\frac{db}{da}$ 倍動き、さらに $y$ が $\frac{dy}{db}$ 倍動く。それらを全部掛け合わせれば「$x$ がちょっと動いたとき $y$ がどれだけ動くか」が求まる。  

ここで、後の実装を見越して $\frac{dy}{dy}$ (自分自身に関する微分) を明示的に含めて書き直す。  

$$ \frac{dy}{dx} = \frac{dy}{dy}\,\frac{dy}{db}\,\frac{db}{da}\,\frac{da}{dx} \tag{5.2} $$

$\frac{dy}{dy}$ は「$y$ が微小変化したとき、$y$ 自身は同じだけ変化する」ので、常に 1 となる。普通は省きくが、ここではあえて残す。  


## 5.2 バックプロパゲーションの導出

式(5.2) は「各微分の積」とだけ言っていて、掛ける順番は決めていない。掛け算なので順番は自由である。そこで、あえて出力側から入力側へという順番でカッコを付ける。  

$$ \frac{dy}{dx} = \left(\left(\frac{dy}{dy}\,\frac{dy}{db}\right)\frac{db}{da}\right)\frac{da}{dx} \tag{5.3} $$

つまり、次の順に計算する。  

1. $\dfrac{dy}{dy}(=1) \times \dfrac{dy}{db} = \dfrac{dy}{db}$
2. その結果 $\times \dfrac{db}{da} = \dfrac{dy}{da}$
3. その結果 $\times \dfrac{da}{dx} = \dfrac{dy}{dx}$ ← 完成！

通常の計算 (順伝播) とは逆向き ($y \to x$ の方向) に微分を伝えていくのが分かる。これがバックプロパゲーション (逆伝播) の名前の由来。  


### 逆伝播の計算グラフ

各関数 $A, B, C$ の導関数を $A'(x), B'(a), C'(b)$ と書くと、$\frac{da}{dx}=A'(x)$、$\frac{db}{da}=B'(a)$、$\frac{dy}{db}=C'(b)$ である。逆伝播の流れを図にすると次のようになる。出力側 (右) の $\frac{dy}{dy}=1$ からスタートし、左へ向かって導関数を掛けていく。  


In [ ]:
# 逆伝播の流れを matplotlib で図示（教科書 図5-3 のイメージを再現）
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.axis('off')

# ノード位置（右から左へ逆伝播）
labels = [r"$dy/dy$" + "\n(=1)", r"$dy/db$", r"$dy/da$", r"$dy/dx$"]
xs = [9, 6, 3, 0]
y0 = 1
for xi, lab in zip(xs, labels):
    ax.add_patch(plt.Circle((xi, y0), 0.55, color="#ffd9a0", ec="#cc8800", zorder=2))
    ax.text(xi, y0, lab, ha="center", va="center", fontsize=11, zorder=3)

# 関数（導関数）ノード（×掛け算）
funcs = [r"$\times\,C'(b)$", r"$\times\,B'(a)$", r"$\times\,A'(x)$"]
fxs = [7.5, 4.5, 1.5]
for xi, lab in zip(fxs, funcs):
    ax.add_patch(plt.Rectangle((xi-0.7, y0-0.4), 1.4, 0.8, color="#a8d8ea", ec="#3a7ca5", zorder=2))
    ax.text(xi, y0, lab, ha="center", va="center", fontsize=10, zorder=3)

# 矢印（右 → 左 = 逆伝播）
for x_start, x_end in [(8.4, 8.2), (6.9, 6.55), (5.4, 5.2), (3.9, 3.55), (2.4, 2.2), (0.9, 0.55)]:
    ax.annotate("", xy=(x_end, y0), xytext=(x_start, y0),
                arrowprops=dict(arrowstyle="->", color="crimson", lw=2))

ax.set_xlim(-1, 10.5)
ax.set_ylim(0, 2)
ax.set_title("Backpropagation: from output (right) to input (left)", fontsize=12)
plt.show()


## なぜ「出力 → 入力」の向きなのか？

順番が自由なら、逆に「入力 → 出力」の向きでもよいはず。なぜ出力から計算するのか？  

理由は、伝播するものを「$y$ の各変数に関する微分」に統一できるからである。式 (5.3) の各段で流れるのは $\frac{dy}{dy}, \frac{dy}{db}, \frac{dy}{da}, \frac{dy}{dx}$ と、すべて「$y$ の ○○ に関する微分」である。$y$ (＝出力、ふつうは損失) を「主役」に固定できます。  

これが機械学習で決定的に重要になる。機械学習の問題は多くの場合、  

$$ \text{大量のパラメータ}\;(x_1, x_2, \ldots, x_N) \;\longrightarrow\; \text{1 つのスカラ値 (損失 } L \text{)} $$

という形をしている。出力 $L$ はたった 1 つ、入力は数百万個。このとき、出力 $L$ から逆向きに一度伝播させるだけで、すべてのパラメータに関する微分 $\frac{dL}{dx_i}$ がまとめて求まる。数値微分のように 1 個ずつ計算する必要がない。この圧倒的な効率の良さが、バックプロパゲーションが使われる理由になる。  


## 5.3 順伝播と逆伝播の対応関係

順伝播と逆伝播を上下に並べると、次のようになる。  

| | 順伝播（通常の計算） | 逆伝播（微分の計算） |
|---|---|---|
| 変数 | $x,\; a,\; b,\; y$ | $\dfrac{dy}{dx},\; \dfrac{dy}{da},\; \dfrac{dy}{db},\; \dfrac{dy}{dy}$ |
| 関数 | $A,\; B,\; C$ (通常の計算) | $A'(x),\; B'(a),\; C'(b)$ (微分の計算) |

つまり、

- 変数には、「通常の値」と「微分の値」の 2 つがある (後の実装で `Variable` が `.data` と `.grad` を持つ理由)  
- 関数には、「順伝播 (forward)」と「逆伝播 (backward)」の 2 つの計算がある  

という構造になっている。これが DeZero（やあらゆる自動微分フレームワーク）の設計の中心。  

### 注意点：逆伝播には順伝播の値が必要

$C'(b)$ を計算するには $b$ の値が要る。$B'(a)$ には $a$ が要る。つまり、逆伝播の計算には、順伝播のときの入力値が必要となる。  

このため実装では、まず順伝播を行い、各関数が自分の入力値を覚えておく必要がある。その後ではじめて逆伝播が計算できる。(次のステップで `Function` が `self.input` を保持するのはこのため)  


### チェインルールを「手計算」と「数値微分」で比較

$y = (e^{x^2})^2$ の微分をチェインルールで計算し、それがステップ 4 の数値微分と一致することを確認する。  

各段の導関数は次のとおり ($a = x^2,\; b = e^{a}$)。

- $\dfrac{da}{dx} = 2x \quad$（$a = x^2$ より）
- $\dfrac{db}{da} = e^{a} = b \quad$（$b = e^{a}$ より）
- $\dfrac{dy}{db} = 2b \quad$（$y = b^2$ より）

チェインルールで掛け合わせると：

$$ \frac{dy}{dx} = \frac{dy}{db}\frac{db}{da}\frac{da}{dx} = (2b)\cdot(b)\cdot(2x) = 4\,b^{2}\,x = 4\,(e^{x^2})^2\,x $$

これを $x = 0.5$ で計算し、数値微分の `3.297...` と比べる。  


In [ ]:
x_val = 0.5

# --- 順伝播（各段の値を覚えておく）---
a = x_val ** 2          # a = x^2
b = np.exp(a)           # b = e^a
y = b ** 2              # y = b^2

# --- 逆伝播（出力側から導関数を掛けていく）---
dy_dy = 1.0             # 出発点: dy/dy = 1
dy_db = dy_dy * (2 * b) # × C'(b) = 2b
dy_da = dy_db * (b)     # × B'(a) = e^a = b
dy_dx = dy_da * (2 * x_val)  # × A'(x) = 2x

print("チェインルール（手計算）: dy/dx =", dy_dx)

# 数値微分と照合
def f_check(xv):
    A, B, C = Square(), Exp(), Square()
    return C(B(A(xv)))
num = numerical_diff(f_check, Variable(np.array(x_val)))
print("数値微分               : dy/dx =", num)
print("差の絶対値             :", abs(dy_dx - num))


ほぼ完全に一致した（わずかな差は数値微分の近似誤差）。チェインルールが正しく「各関数の局所的な微分の積」で全体の微分を与えていることが、手を動かして確認できた。

> ステップ 5 のまとめ
> - チェインルール: 合成関数の微分 = 各関数の局所的な微分の積。
> - その積を **出力 → 入力** の順に計算するのがバックプロパゲーション。伝播するのは常に「出力 $y$ の各変数に関する微分」。
> - 出力が1つ・入力が大量、というニューラルネットの構造で **一度の逆伝播で全パラメータの微分が求まる** ため、圧倒的に効率が良い。
> - 逆伝播には順伝播時の値が必要 → 変数は値と微分の2つ、関数は forward と backward の2つを持つ。


---

# 今回のまとめ

第1ステージ（ステップ1〜5）で、自動微分フレームワークの土台ができた。

1. ステップ 1 `Variable`: データを入れる「箱」。
2. ステップ 2 `Function`: 箱を受け取り計算して箱を返す基底クラス。共通処理（`__call__`）と個別計算（`forward`）を分離。
3. ステップ 3 連結: 入出力を `Variable` で統一したことで関数を数珠つなぎにでき、合成関数が作れる。
4. ステップ 4 数値微分: 微分の定義から近似的に微分を計算。簡単だが「遅い・誤差が出る」。検算用に有用。
5. ステップ 5 バックプロパゲーション理論: チェインルールを「出力→入力」の順に適用。出力1つ・入力多数のニューラルネットで一度の逆伝播で全微分が求まる。これが PyTorch などの中心。

次のステージ（ステップ6以降）では、この理論を実際にコードに落とし込み、`y.backward()` で本当に自動微分が動くところまで作っていく。今日学んだ「変数は値と微分を持つ」「関数は forward と backward を持つ」「逆伝播には順伝播の値が必要」という3点が、その実装の設計図になる。
